In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
baseline_path = Path("/mnt/baseline/summary_baseline.json")
retrieval_path = Path("/logs/retrieval/summary_retrieval.json")
langmem_path = Path("/logs/langmem/summary_langmem.json")


with open(baseline_path, "r") as f:
    baseline_data = json.load(f)
with open(retrieval_path, "r") as f:
    retrieval_data = json.load(f)
with open(langmem_path, "r") as f:
    langmem_data = json.load(f)

baseline_df = pd.DataFrame(baseline_data)
retrieval_df = pd.DataFrame(retrieval_data)
langmem_df = pd.DataFrame(langmem_data)


baseline_df["agent_type"] = "baseline"
retrieval_df["agent_type"] = "retrieval"
langmem_df["agent_type"] = "retrieval"

# Merge
combined_df = pd.concat([baseline_df, retrieval_df, langmem_df], ignore_index=True)

# Ordered difficulty
combined_df["difficulty"] = pd.Categorical(
    combined_df["difficulty"],
    categories=["easy", "medium", "challenging", "hard", "very_hard"],
    ordered=True
)

In [ ]:


grouped_means = combined_df.groupby(["agent_type", "difficulty"]).agg({
    "score": "mean",
    "steps_taken": "mean",
    "total_tokens": "mean",
    "duration_seconds": "mean",
    "won": "mean"
}).reset_index().rename(columns={"won": "win_rate"})

difficulty_coverage = combined_df.groupby(["agent_type", "difficulty"]).size().unstack(fill_value=0)

grouped_means, difficulty_coverage

# Group by agent and difficulty to get average move efficiency
efficiency_df = combined_df.groupby(["agent", "difficulty"]).agg(
    avg_move_efficiency=("move_efficiency", "mean"),
    avg_moves=("moves", "mean"),
    avg_steps=("steps_taken", "mean")
).reset_index()

avg_moves_df = combined_df.groupby(["agent", "difficulty"]).agg(
    avg_moves=("moves", "mean"),
    avg_steps=("steps_taken", "mean"),
    avg_score=("score", "mean"),
    win_rate=("won", "mean")
).reset_index()

# Add success/failure label for move distribution
combined_df["result"] = combined_df["won"].map({True: "Success", False: "Failure"})

# Filter only successful episodes for histogram
success_df = combined_df[combined_df["won"] == True]


In [ ]:

sns.set(style="whitegrid")

# --- 1. Average Score per Difficulty Level by Agent ---
fig1, ax1 = plt.subplots(figsize=(10, 6))
sns.barplot(data=combined_df, x="difficulty", y="score", hue="agent", ci=None, ax=ax1)
ax1.set_title("Average Score per Difficulty Level by Agent")
ax1.set_ylabel("Average Score")
fig1_code = """
sns.barplot(data=combined_df, x="difficulty", y="score", hue="agent", ci=None)
plt.title("Average Score per Difficulty Level by Agent")
plt.ylabel("Average Score")
"""

# --- 2. Success Rate vs Difficulty ---
fig2, ax2 = plt.subplots(figsize=(10, 6))
sns.lineplot(data=combined_df, x="difficulty", y="won", hue="agent", estimator="mean", marker="o", ax=ax2)
ax2.set_title("Success Rate vs. Difficulty")
ax2.set_ylabel("Success Rate")
fig2_code = """
sns.lineplot(data=combined_df, x="difficulty", y="won", hue="agent", estimator="mean", marker="o")
plt.title("Success Rate vs. Difficulty")
plt.ylabel("Success Rate")
"""

# --- 3. Step Count to Success (Only Successful Episodes) ---
fig3, ax3 = plt.subplots(figsize=(10, 6))
sns.histplot(data=success_df, x="steps_taken", hue="agent", multiple="stack", bins=20, ax=ax3)
ax3.set_title("Step Count to Success (Only Successful Episodes)")
ax3.set_xlabel("Steps Taken")
fig3_code = """
success_df = combined_df[combined_df["won"] == True]
sns.histplot(data=success_df, x="steps_taken", hue="agent", multiple="stack", bins=20)
plt.title("Step Count to Success (Only Successful Episodes)")
"""

# --- 4. Average Moves per Episode by Difficulty and Agent ---
fig4, ax4 = plt.subplots(figsize=(10, 6))
sns.barplot(data=avg_moves_df, x="difficulty", y="avg_moves", hue="agent", ax=ax4)
ax4.set_title("Average Moves per Episode by Difficulty and Agent")
ax4.set_ylabel("Average Moves")
fig4_code = """
sns.barplot(data=avg_moves_df, x="difficulty", y="avg_moves", hue="agent")
plt.title("Average Moves per Episode by Difficulty and Agent")
plt.ylabel("Average Moves")
"""

# --- 5. Distribution of Moves by Result (Success/Failure) ---
fig5, ax5 = plt.subplots(figsize=(10, 6))
sns.boxplot(data=combined_df, x="difficulty", y="moves", hue="result", ax=ax5)
ax5.set_title("Distribution of Moves by Result (Success/Failure)")
ax5.set_ylabel("Number of Moves")
fig5_code = """
sns.boxplot(data=combined_df, x="difficulty", y="moves", hue="result")
plt.title("Distribution of Moves by Result (Success/Failure)")
plt.ylabel("Number of Moves")
"""

# --- 6. Moves-to-Steps Efficiency Ratio by Agent and Difficulty ---
fig6, ax6 = plt.subplots(figsize=(10, 6))
sns.barplot(data=efficiency_df, x="difficulty", y="avg_move_efficiency", hue="agent", ax=ax6)
ax6.set_title("Moves-to-Steps Efficiency Ratio by Agent and Difficulty")
ax6.set_ylabel("Move Efficiency (Moves / Steps Taken)")
fig6_code = """
sns.barplot(data=efficiency_df, x="difficulty", y="avg_move_efficiency", hue="agent")
plt.title("Moves-to-Steps Efficiency Ratio by Agent and Difficulty")
plt.ylabel("Move Efficiency (Moves / Steps Taken)")
"""

plt.tight_layout()
plt.show()

